# Compare Two Rollouts

Run a single rollout from each of two checkpoints at a chosen fixed logvar, then render each as a separate video.

In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import time
from pathlib import Path

import jax
import jax.numpy as jnp
import mediapy as media
import mujoco
import numpy as np
import orbax.checkpoint as ocp
from jax import random
from omegaconf import OmegaConf

from scamper.environment.rodent import consts as rodent_consts

from scamper.agent.distill.distill_networks import (
    create_abstract_distill_policy,
)
from scamper.agent.mlp_prior.prior_networks import (
    create_abstract_prior_policy,
)
from scamper.agent.observation_utils import flatten_obs_dict
from scamper.scripts.multi_prior_rollout import (
    create_clip_state,
    create_environment,
    create_prior_policy,
    fix_paths,
)

In [ ]:
# ============ CHECKPOINT PATHS ============
CHECKPOINT_A_PATH = (
    "/home/mila/a/aidan.sirbu/SCAMPER/scratch/prior_checkpoints/golden_rodent_prior_jax"
)
CHECKPOINT_A_STEP = None  # None = latest
CHECKPOINT_A_LABEL = "Prior"
CHECKPOINT_A_TYPE = "prior"  # "prior" or "distill"

CHECKPOINT_B_PATH = (
    "/home/mila/a/aidan.sirbu/SCAMPER/scratch/distill_checkpoints/260327_222349_298131"
)
CHECKPOINT_B_STEP = None
CHECKPOINT_B_LABEL = "Distill"
CHECKPOINT_B_TYPE = "distill"  # "prior" or "distill"

# ============ ROLLOUT SETTINGS ============
FIXED_LOGVAR = 0.0  # Single logvar value to use for both rollouts
MAX_STEPS = 1000  # Maximum steps per rollout
CLIP_INDICES = [550, 551, 552, 553]  # List of reference clip indices to roll out
START_FRAME = 0  # Frame within clip (None = random)

# ============ RENDERING ============
RENDER_FPS = 50
CAMERA_NAME = "close_profile-rodent"
RENDER_HEIGHT = 480
RENDER_WIDTH = 640

# ============ RANDOM SEED ============
SEED = 42

## Load Checkpoints

In [ ]:
def load_checkpoint(
    checkpoint_path: str, step: int | None = None, checkpoint_type: str = "prior"
):
    """Load a prior or distill checkpoint.

    Returns:
        Tuple of (cfg, policy_params, step).
    """
    if checkpoint_type == "prior":
        step_prefix = "PriorNetwork"
    elif checkpoint_type == "distill":
        step_prefix = "DistillNetwork"
    else:
        raise ValueError(f"Unknown checkpoint_type: {checkpoint_type}")

    mgr_options = ocp.CheckpointManagerOptions(create=False, step_prefix=step_prefix)

    with ocp.CheckpointManager(checkpoint_path, options=mgr_options) as ckpt_mgr:
        if step is None:
            step = ckpt_mgr.latest_step()
        print(
            f"Loading {checkpoint_type} checkpoint from {checkpoint_path} at step {step}"
        )

        cfg = ckpt_mgr.restore(
            step,
            args=ocp.args.Composite(config=ocp.args.JsonRestore()),
        )["config"]
        cfg = OmegaConf.create(cfg)

        if checkpoint_type == "prior":
            prior_hidden_layer_sizes = tuple(
                cfg.network_config.get(
                    "prior_layer_sizes", cfg.network_config.encoder_layer_sizes
                )
            )
            abstract_policy = create_abstract_prior_policy(
                cfg=OmegaConf.to_container(cfg),
                prior_hidden_layer_sizes=prior_hidden_layer_sizes,
            )
        else:  # distill
            obs_sizes = dict(cfg.network_config.obs_sizes)
            abstract_policy = create_abstract_distill_policy(
                obs_sizes=obs_sizes,
                latent_size=int(cfg.network_config.intention_size),
                action_size=int(cfg.network_config.action_size),
                encoder_hidden_layer_sizes=tuple(
                    cfg.network_config.encoder_layer_sizes
                ),
                decoder_hidden_layer_sizes=tuple(
                    cfg.network_config.decoder_layer_sizes
                ),
                prior_hidden_layer_sizes=tuple(cfg.network_config.prior_layer_sizes),
            )

        policy = ckpt_mgr.restore(
            step,
            args=ocp.args.Composite(policy=ocp.args.StandardRestore(abstract_policy)),
        )["policy"]

    cfg = fix_paths(cfg)
    return cfg, policy, step


cfg_a, policy_params_a, step_a = load_checkpoint(
    CHECKPOINT_A_PATH, CHECKPOINT_A_STEP, CHECKPOINT_A_TYPE
)
print(
    f"  [{CHECKPOINT_A_LABEL}] step={step_a}, action_size={cfg_a.network_config.action_size}, "
    f"intention_size={cfg_a.network_config.intention_size}\n"
)

cfg_b, policy_params_b, step_b = load_checkpoint(
    CHECKPOINT_B_PATH, CHECKPOINT_B_STEP, CHECKPOINT_B_TYPE
)
print(
    f"  [{CHECKPOINT_B_LABEL}] step={step_b}, action_size={cfg_b.network_config.action_size}, "
    f"intention_size={cfg_b.network_config.intention_size}"
)

## Create Environment (Warp + No-Tail)

In [ ]:
env_cfg = OmegaConf.create(OmegaConf.to_container(cfg_a, resolve=True))
OmegaConf.set_struct(env_cfg.env_config, False)
OmegaConf.update(env_cfg.env_config, "mujoco_impl", "warp")
OmegaConf.update(
    env_cfg.env_config,
    "walker_xml_path",
    str(rodent_consts.RODENT_NO_TAIL_COLLISION_XML),
)
OmegaConf.set_struct(env_cfg.env_config, True)

print(f"Backend: {env_cfg.env_config.mujoco_impl}")
print(f"Walker XML: {env_cfg.env_config.walker_xml_path}")

env = create_environment(env_cfg)

# Access reference clips for sampling starting poses
unwrapped_env = env
while hasattr(unwrapped_env, "_env"):
    unwrapped_env = unwrapped_env._env

reference_clips = unwrapped_env.reference_clips
num_clips = reference_clips.qpos.shape[0]
clip_length = reference_clips.qpos.shape[1]

print(f"Reference clips: {num_clips} clips, {clip_length} frames each")
print(f"Action size: {env.action_size}")

## Create Policy Functions and Run Rollouts

In [ ]:
def get_network_params(cfg):
    """Extract network architecture params from a checkpoint config."""
    nc = cfg.network_config
    action_size = nc.action_size
    intention_size = nc.intention_size
    decoder_layers = tuple(nc.decoder_layer_sizes)
    prior_layers = tuple(nc.get("prior_layer_sizes", nc.encoder_layer_sizes))

    if "obs_sizes" in nc:
        proprio_obs_size = nc.obs_sizes["proprioception"]
    elif "proprioceptive_obs_size" in nc:
        proprio_obs_size = nc.proprioceptive_obs_size
    else:
        proprio_obs_size = nc.observation_size - nc.get("reference_obs_size", 0)

    return action_size, intention_size, decoder_layers, prior_layers, proprio_obs_size


def make_policy(cfg, policy_params, fixed_logvar):
    """Create a prior-only policy function from a checkpoint and logvar setting."""
    action_size, intention_size, decoder_layers, prior_layers, proprio_obs_size = (
        get_network_params(cfg)
    )
    return create_prior_policy(
        policy_params=policy_params,
        action_size=action_size,
        proprioceptive_obs_size=proprio_obs_size,
        intention_latent_size=intention_size,
        decoder_hidden_layer_sizes=decoder_layers,
        prior_hidden_layer_sizes=prior_layers,
        fixed_logvar=fixed_logvar,
        deterministic=False,
    )


policy_fn_a = make_policy(cfg_a, policy_params_a, FIXED_LOGVAR)
policy_fn_b = make_policy(cfg_b, policy_params_b, FIXED_LOGVAR)

print(f"Logvar: {FIXED_LOGVAR}")


def create_rollout_fn(env, policy_fn, max_steps):
    """Create a JIT-compiled rollout fn that takes (starting_state, rng_key) as args."""
    jit_step = jax.jit(env.step)

    @jax.jit
    def rollout_fn(starting_state, rng_key):
        state = starting_state

        def step_fn(carry, _):
            state, key = carry
            key, key_action = random.split(key)

            if hasattr(state.obs, "get") or isinstance(state.obs, dict):
                flat_obs = flatten_obs_dict(state.obs)
                proprio = flat_obs["proprioception"]
            else:
                proprio = state.obs

            action, _ = policy_fn(proprio, key_action)
            next_state = jit_step(state, action)

            return (next_state, key), next_state

        initial_carry = (state, rng_key)
        _, all_states = jax.lax.scan(step_fn, initial_carry, None, length=max_steps)

        return all_states

    return rollout_fn


# Build rollout functions once per model
rollout_fn_a = create_rollout_fn(env, policy_fn_a, MAX_STEPS)
rollout_fn_b = create_rollout_fn(env, policy_fn_b, MAX_STEPS)

models = [
    (CHECKPOINT_A_LABEL, rollout_fn_a),
    (CHECKPOINT_B_LABEL, rollout_fn_b),
]

# Roll out each clip index for each model
rng_np = np.random.default_rng(SEED)
jax_rng = random.PRNGKey(SEED)

# qpos_results[(clip_idx, label)] = qpos array
qpos_results = {}

for clip_idx in CLIP_INDICES:
    start_frame = START_FRAME if START_FRAME is not None else int(rng_np.integers(0, clip_length))
    jax_rng, rng_init, rng_rollout = random.split(jax_rng, 3)
    starting_state = create_clip_state(env, clip_idx, start_frame, rng_init)
    print(f"\n=== Clip {clip_idx}, frame={start_frame} ===")

    for label, rollout_fn in models:
        print(f"  Running [{label}]...", end=" ")
        t0 = time.time()
        states = rollout_fn(starting_state, rng_rollout)
        states.data.qpos.block_until_ready()
        qpos_results[(clip_idx, label)] = np.array(states.data.qpos)
        del states
        print(f"Done in {time.time() - t0:.1f}s")

## Render Videos

In [ ]:
def render_qpos_trajectory(
    qpos_trajectory, mj_model, camera_name, height=480, width=640
):
    """Render frames directly from a qpos trajectory using MuJoCo's C renderer."""
    mj_data = mujoco.MjData(mj_model)
    renderer = mujoco.Renderer(mj_model, height=height, width=width)

    frames = []
    for qpos in qpos_trajectory:
        mj_data.qpos[:] = qpos
        mujoco.mj_forward(mj_model, mj_data)
        renderer.update_scene(mj_data, camera=camera_name)
        frames.append(renderer.render().copy())

    renderer.close()
    return frames


mj_model = env.mj_model

for clip_idx in CLIP_INDICES:
    for label in [CHECKPOINT_A_LABEL, CHECKPOINT_B_LABEL]:
        qpos = qpos_results[(clip_idx, label)]
        title = f"{label} — clip {clip_idx} (logvar={FIXED_LOGVAR})"
        print(f"Rendering [{title}] ({qpos.shape[0]} frames)...")
        t0 = time.time()
        frames = render_qpos_trajectory(
            qpos, mj_model, CAMERA_NAME, RENDER_HEIGHT, RENDER_WIDTH
        )
        print(f"  Done in {time.time() - t0:.1f}s")
        media.show_video(frames, fps=RENDER_FPS, title=title)

## Interactive Trail Viewer

Select a clip index and render an interactive motion trail for each checkpoint.

In [31]:
import ipywidgets as widgets
from IPython.display import display as ipy_display
import io
import PIL.Image

# --- Configuration: Checkpoint A ---
RENDER_CLIP_INDEX_A = 550
TRAIL_TIMESTEPS_A = [0, 200, 450, 600, 770, 900]  # Explicit timesteps; empty list = use linspace
TRAIL_NUM_FRAMES_A = 10  # Number of poses if TRAIL_TIMESTEPS is empty
TRAIL_T_START_A = 0
TRAIL_T_END_A = 999
TRAIL_LINE_STEP_A = 5  # Connect torso positions every N timesteps
TRAIL_LINE_RADIUS_A = 0.003
TRAIL_LINE_RGB_A = (189, 19, 0)
TRAIL_LINE_ALPHA_A = 0.8
TRAIL_CAM_POS_A = [-0.7, 0.1, 0.25]
TRAIL_CAM_LOOKAT_A = [0.0, 0.425, 0.0]
TRAIL_INTERACTIVE_HEIGHT_A = 720
TRAIL_INTERACTIVE_WIDTH_A = 1280

# --- Configuration: Checkpoint B ---
RENDER_CLIP_INDEX_B = 550
TRAIL_TIMESTEPS_B = [0, 200, 450, 600, 770, 900]
TRAIL_NUM_FRAMES_B = 10
TRAIL_T_START_B = 0
TRAIL_T_END_B = 999
TRAIL_LINE_STEP_B = 5
TRAIL_LINE_RADIUS_B = 0.003
TRAIL_LINE_RGB_B = (189, 19, 0)
TRAIL_LINE_ALPHA_B = 0.8
TRAIL_CAM_POS_B = [-0.7, 0.1, 0.25]
TRAIL_CAM_LOOKAT_B = [0.0, 0.425, 0.0]
TRAIL_INTERACTIVE_HEIGHT_B = 720
TRAIL_INTERACTIVE_WIDTH_B = 1280

# --- Build single-rodent model (for torso position computation) ---
walker_xml = str(rodent_consts.RODENT_NO_TAIL_COLLISION_XML)
arena_xml = str(rodent_consts.ARENA_XML_PATH)

_arena_spec = mujoco.MjSpec.from_file(arena_xml)
_rodent_spec = mujoco.MjSpec.from_file(walker_xml)
_frame = _arena_spec.worldbody.add_frame(pos=(0, 0, 0), quat=(1, 0, 0, 0))
_body = _frame.attach_body(_rodent_spec.body("walker"), "", suffix="-rodent")
_body.add_freejoint(name="root")
single_model = _arena_spec.compile()
single_data = mujoco.MjData(single_model)
torso_body_id = mujoco.mj_name2id(single_model, mujoco.mjtObj.mjOBJ_BODY, "torso-rodent")
nq_single = single_model.nq

print(f"Single-rodent model: nq={nq_single}, torso body id={torso_body_id}")
print(f"Render clip index A: {RENDER_CLIP_INDEX_A}, B: {RENDER_CLIP_INDEX_B}")

Single-rodent model: nq=74, torso body id=3
Render clip index A: 550, B: 550


In [27]:
# --- Interactive trail viewer: Checkpoint A ---
qpos_a = qpos_results[(RENDER_CLIP_INDEX_A, CHECKPOINT_A_LABEL)]

# Compute trail indices
if TRAIL_TIMESTEPS_A:
    trail_indices_a = np.array(TRAIL_TIMESTEPS_A, dtype=int)
else:
    trail_indices_a = np.linspace(TRAIL_T_START_A, TRAIL_T_END_A, TRAIL_NUM_FRAMES_A, dtype=int)
trail_line_rgba_a = [c / 255 for c in TRAIL_LINE_RGB_A] + [TRAIL_LINE_ALPHA_A]

# Compute torso positions for trajectory trace line
line_timesteps_a = np.arange(trail_indices_a[0], trail_indices_a[-1] + 1, TRAIL_LINE_STEP_A)
torso_positions_a = []
for t in line_timesteps_a:
    single_data.qpos[:] = np.array(qpos_a[t])
    mujoco.mj_forward(single_model, single_data)
    torso_positions_a.append(single_data.xpos[torso_body_id].copy())

# Build multi-rodent model
trail_spec_a = mujoco.MjSpec.from_file(arena_xml)
for i in range(len(trail_indices_a)):
    rodent_spec_i = mujoco.MjSpec.from_file(walker_xml)
    frame_i = trail_spec_a.worldbody.add_frame(pos=(0, 0, 0), quat=(1, 0, 0, 0))
    body_i = frame_i.attach_body(rodent_spec_i.body("walker"), "", suffix=f"-t{i}")
    body_i.add_freejoint(name=f"root-t{i}")

# Add trajectory line segments
for j in range(len(torso_positions_a) - 1):
    p1 = torso_positions_a[j].tolist()
    p2 = torso_positions_a[j + 1].tolist()
    trail_spec_a.worldbody.add_geom(
        name=f"line_seg_{j}",
        type=mujoco.mjtGeom.mjGEOM_CAPSULE,
        fromto=[*p1, *p2],
        size=[TRAIL_LINE_RADIUS_A, 0, 0],
        rgba=trail_line_rgba_a,
        contype=0,
        conaffinity=0,
    )

# Add overhead light
overhead_a = trail_spec_a.worldbody.add_light(name="overhead")
overhead_a.pos = [0.0, 0.425, 3.0]
overhead_a.dir = [0, 0, -1]
overhead_a.diffuse = [1.0, 1.0, 1.0]
overhead_a.specular = [0.3, 0.3, 0.3]

trail_model_a = trail_spec_a.compile()

# Disable tracking lights
for i in range(trail_model_a.nlight):
    name = mujoco.mj_id2name(trail_model_a, mujoco.mjtObj.mjOBJ_LIGHT, i)
    if name and "tracking_light" in name:
        trail_model_a.light_diffuse[i] = [0, 0, 0]
        trail_model_a.light_specular[i] = [0, 0, 0]

trail_data_a = mujoco.MjData(trail_model_a)

# Set each copy's qpos to a different timestep
for i, t in enumerate(trail_indices_a):
    start = i * nq_single
    trail_data_a.qpos[start : start + nq_single] = np.array(qpos_a[t])
mujoco.mj_forward(trail_model_a, trail_data_a)

print(
    f"[{CHECKPOINT_A_LABEL}] Trail with {len(trail_indices_a)} poses at "
    f"timesteps: {trail_indices_a.tolist()}"
)
print(f"  {len(torso_positions_a) - 1} line segments (every {TRAIL_LINE_STEP_A} steps)")

# --- Interactive viewer ---
# Persist captured camera across re-runs
try:
    captured_cam_a
except NameError:
    captured_cam_a = {}

renderer_a = mujoco.Renderer(
    trail_model_a, height=TRAIL_INTERACTIVE_HEIGHT_A, width=TRAIL_INTERACTIVE_WIDTH_A
)
free_cam_a = mujoco.MjvCamera()
free_cam_a.type = mujoco.mjtCamera.mjCAMERA_FREE

if captured_cam_a:
    free_cam_a.azimuth = captured_cam_a["azimuth"]
    free_cam_a.elevation = captured_cam_a["elevation"]
    free_cam_a.distance = captured_cam_a["distance"]
    free_cam_a.lookat[:] = captured_cam_a["lookat"]
else:
    free_cam_a.lookat[:] = TRAIL_CAM_LOOKAT_A
    _offset_a = np.array(TRAIL_CAM_POS_A) - np.array(TRAIL_CAM_LOOKAT_A)
    free_cam_a.distance = float(np.linalg.norm(_offset_a))
    free_cam_a.azimuth = float(np.degrees(np.arctan2(_offset_a[1], _offset_a[0])))
    free_cam_a.elevation = float(
        np.degrees(np.arcsin(_offset_a[2] / np.linalg.norm(_offset_a)))
    )

img_widget_a = widgets.Image(
    format="jpeg", layout=widgets.Layout(width=f"{TRAIL_INTERACTIVE_WIDTH_A}px")
)


def render_view_a(change=None):
    renderer_a.update_scene(trail_data_a, camera=free_cam_a)
    frame = renderer_a.render()
    buf = io.BytesIO()
    PIL.Image.fromarray(frame).save(buf, format="JPEG", quality=90)
    img_widget_a.value = buf.getvalue()


azimuth_a = widgets.FloatSlider(
    value=free_cam_a.azimuth, min=-180, max=180, step=1,
    description="Azimuth", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
elevation_a = widgets.FloatSlider(
    value=free_cam_a.elevation, min=-90, max=90, step=1,
    description="Elevation", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
distance_a = widgets.FloatSlider(
    value=free_cam_a.distance, min=0.1, max=5.0, step=0.05,
    description="Distance", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
lookat_x_a = widgets.FloatSlider(
    value=free_cam_a.lookat[0], min=-3, max=3, step=0.05,
    description="Lookat X", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
lookat_y_a = widgets.FloatSlider(
    value=free_cam_a.lookat[1], min=-3, max=3, step=0.05,
    description="Lookat Y", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
lookat_z_a = widgets.FloatSlider(
    value=free_cam_a.lookat[2], min=-1, max=2, step=0.05,
    description="Lookat Z", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)


def on_slider_change_a(change):
    free_cam_a.azimuth = azimuth_a.value
    free_cam_a.elevation = elevation_a.value
    free_cam_a.distance = distance_a.value
    free_cam_a.lookat[:] = [lookat_x_a.value, lookat_y_a.value, lookat_z_a.value]
    render_view_a()


for s in [azimuth_a, elevation_a, distance_a, lookat_x_a, lookat_y_a, lookat_z_a]:
    s.observe(on_slider_change_a, names="value")

capture_label_a = widgets.Label(value="")
capture_btn_a = widgets.Button(description="Capture Camera", button_style="success")


def on_capture_a(btn):
    captured_cam_a["azimuth"] = azimuth_a.value
    captured_cam_a["elevation"] = elevation_a.value
    captured_cam_a["distance"] = distance_a.value
    captured_cam_a["lookat"] = [lookat_x_a.value, lookat_y_a.value, lookat_z_a.value]
    capture_label_a.value = (
        f"Captured: az={captured_cam_a['azimuth']:.1f} "
        f"el={captured_cam_a['elevation']:.1f} "
        f"dist={captured_cam_a['distance']:.2f} "
        f"lookat={[f'{v:.2f}' for v in captured_cam_a['lookat']]}"
    )


capture_btn_a.on_click(on_capture_a)
render_view_a()

title_a = widgets.HTML(
    value=f"<h3>{CHECKPOINT_A_LABEL} — clip {RENDER_CLIP_INDEX_A}</h3>"
)
controls_a = widgets.VBox([
    title_a,
    img_widget_a,
    widgets.HBox([azimuth_a, elevation_a]),
    widgets.HBox([distance_a]),
    widgets.HBox([lookat_x_a, lookat_y_a, lookat_z_a]),
    widgets.HBox([capture_btn_a, capture_label_a]),
])
ipy_display(controls_a)

[Prior] Trail with 6 poses at timesteps: [0, 200, 450, 600, 770, 900]
  180 line segments (every 5 steps)


In [32]:
# --- Interactive trail viewer: Checkpoint B ---
qpos_b = qpos_results[(RENDER_CLIP_INDEX_B, CHECKPOINT_B_LABEL)]

# Compute trail indices
if TRAIL_TIMESTEPS_B:
    trail_indices_b = np.array(TRAIL_TIMESTEPS_B, dtype=int)
else:
    trail_indices_b = np.linspace(TRAIL_T_START_B, TRAIL_T_END_B, TRAIL_NUM_FRAMES_B, dtype=int)
trail_line_rgba_b = [c / 255 for c in TRAIL_LINE_RGB_B] + [TRAIL_LINE_ALPHA_B]

# Compute torso positions for trajectory trace line
line_timesteps_b = np.arange(trail_indices_b[0], trail_indices_b[-1] + 1, TRAIL_LINE_STEP_B)
torso_positions_b = []
for t in line_timesteps_b:
    single_data.qpos[:] = np.array(qpos_b[t])
    mujoco.mj_forward(single_model, single_data)
    torso_positions_b.append(single_data.xpos[torso_body_id].copy())

# Build multi-rodent model
trail_spec_b = mujoco.MjSpec.from_file(arena_xml)
for i in range(len(trail_indices_b)):
    rodent_spec_i = mujoco.MjSpec.from_file(walker_xml)
    frame_i = trail_spec_b.worldbody.add_frame(pos=(0, 0, 0), quat=(1, 0, 0, 0))
    body_i = frame_i.attach_body(rodent_spec_i.body("walker"), "", suffix=f"-t{i}")
    body_i.add_freejoint(name=f"root-t{i}")

# Add trajectory line segments
for j in range(len(torso_positions_b) - 1):
    p1 = torso_positions_b[j].tolist()
    p2 = torso_positions_b[j + 1].tolist()
    trail_spec_b.worldbody.add_geom(
        name=f"line_seg_{j}",
        type=mujoco.mjtGeom.mjGEOM_CAPSULE,
        fromto=[*p1, *p2],
        size=[TRAIL_LINE_RADIUS_B, 0, 0],
        rgba=trail_line_rgba_b,
        contype=0,
        conaffinity=0,
    )

# Add overhead light
overhead_b = trail_spec_b.worldbody.add_light(name="overhead")
overhead_b.pos = [0.0, 0.425, 3.0]
overhead_b.dir = [0, 0, -1]
overhead_b.diffuse = [1.0, 1.0, 1.0]
overhead_b.specular = [0.3, 0.3, 0.3]

trail_model_b = trail_spec_b.compile()

# Disable tracking lights
for i in range(trail_model_b.nlight):
    name = mujoco.mj_id2name(trail_model_b, mujoco.mjtObj.mjOBJ_LIGHT, i)
    if name and "tracking_light" in name:
        trail_model_b.light_diffuse[i] = [0, 0, 0]
        trail_model_b.light_specular[i] = [0, 0, 0]

trail_data_b = mujoco.MjData(trail_model_b)

# Set each copy's qpos to a different timestep
for i, t in enumerate(trail_indices_b):
    start = i * nq_single
    trail_data_b.qpos[start : start + nq_single] = np.array(qpos_b[t])
mujoco.mj_forward(trail_model_b, trail_data_b)

print(
    f"[{CHECKPOINT_B_LABEL}] Trail with {len(trail_indices_b)} poses at "
    f"timesteps: {trail_indices_b.tolist()}"
)
print(f"  {len(torso_positions_b) - 1} line segments (every {TRAIL_LINE_STEP_B} steps)")

# --- Interactive viewer ---
# Persist captured camera across re-runs
try:
    captured_cam_b
except NameError:
    captured_cam_b = {}

renderer_b = mujoco.Renderer(
    trail_model_b, height=TRAIL_INTERACTIVE_HEIGHT_B, width=TRAIL_INTERACTIVE_WIDTH_B
)
free_cam_b = mujoco.MjvCamera()
free_cam_b.type = mujoco.mjtCamera.mjCAMERA_FREE

if captured_cam_b:
    free_cam_b.azimuth = captured_cam_b["azimuth"]
    free_cam_b.elevation = captured_cam_b["elevation"]
    free_cam_b.distance = captured_cam_b["distance"]
    free_cam_b.lookat[:] = captured_cam_b["lookat"]
else:
    free_cam_b.lookat[:] = TRAIL_CAM_LOOKAT_B
    _offset_b = np.array(TRAIL_CAM_POS_B) - np.array(TRAIL_CAM_LOOKAT_B)
    free_cam_b.distance = float(np.linalg.norm(_offset_b))
    free_cam_b.azimuth = float(np.degrees(np.arctan2(_offset_b[1], _offset_b[0])))
    free_cam_b.elevation = float(
        np.degrees(np.arcsin(_offset_b[2] / np.linalg.norm(_offset_b)))
    )

img_widget_b = widgets.Image(
    format="jpeg", layout=widgets.Layout(width=f"{TRAIL_INTERACTIVE_WIDTH_B}px")
)


def render_view_b(change=None):
    renderer_b.update_scene(trail_data_b, camera=free_cam_b)
    frame = renderer_b.render()
    buf = io.BytesIO()
    PIL.Image.fromarray(frame).save(buf, format="JPEG", quality=90)
    img_widget_b.value = buf.getvalue()


azimuth_b = widgets.FloatSlider(
    value=free_cam_b.azimuth, min=-180, max=180, step=1,
    description="Azimuth", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
elevation_b = widgets.FloatSlider(
    value=free_cam_b.elevation, min=-90, max=90, step=1,
    description="Elevation", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
distance_b = widgets.FloatSlider(
    value=free_cam_b.distance, min=0.1, max=5.0, step=0.05,
    description="Distance", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
lookat_x_b = widgets.FloatSlider(
    value=free_cam_b.lookat[0], min=-3, max=3, step=0.05,
    description="Lookat X", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
lookat_y_b = widgets.FloatSlider(
    value=free_cam_b.lookat[1], min=-3, max=3, step=0.05,
    description="Lookat Y", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
lookat_z_b = widgets.FloatSlider(
    value=free_cam_b.lookat[2], min=-1, max=2, step=0.05,
    description="Lookat Z", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)


def on_slider_change_b(change):
    free_cam_b.azimuth = azimuth_b.value
    free_cam_b.elevation = elevation_b.value
    free_cam_b.distance = distance_b.value
    free_cam_b.lookat[:] = [lookat_x_b.value, lookat_y_b.value, lookat_z_b.value]
    render_view_b()


for s in [azimuth_b, elevation_b, distance_b, lookat_x_b, lookat_y_b, lookat_z_b]:
    s.observe(on_slider_change_b, names="value")

capture_label_b = widgets.Label(value="")
capture_btn_b = widgets.Button(description="Capture Camera", button_style="success")


def on_capture_b(btn):
    captured_cam_b["azimuth"] = azimuth_b.value
    captured_cam_b["elevation"] = elevation_b.value
    captured_cam_b["distance"] = distance_b.value
    captured_cam_b["lookat"] = [lookat_x_b.value, lookat_y_b.value, lookat_z_b.value]
    capture_label_b.value = (
        f"Captured: az={captured_cam_b['azimuth']:.1f} "
        f"el={captured_cam_b['elevation']:.1f} "
        f"dist={captured_cam_b['distance']:.2f} "
        f"lookat={[f'{v:.2f}' for v in captured_cam_b['lookat']]}"
    )


capture_btn_b.on_click(on_capture_b)
render_view_b()

title_b = widgets.HTML(
    value=f"<h3>{CHECKPOINT_B_LABEL} — clip {RENDER_CLIP_INDEX_B}</h3>"
)
controls_b = widgets.VBox([
    title_b,
    img_widget_b,
    widgets.HBox([azimuth_b, elevation_b]),
    widgets.HBox([distance_b]),
    widgets.HBox([lookat_x_b, lookat_y_b, lookat_z_b]),
    widgets.HBox([capture_btn_b, capture_label_b]),
])
ipy_display(controls_b)

[Distill] Trail with 6 poses at timesteps: [0, 200, 450, 600, 770, 900]
  180 line segments (every 5 steps)


## Export Captured Views

Render both captured camera views side-by-side and save as SVG, PDF, and PNG.

In [35]:
import matplotlib.pyplot as plt
from pathlib import Path

# --- Configuration ---
EXPORT_DIR = Path("/home/mila/a/aidan.sirbu/SCAMPER/notebooks/plots/random_generation")
EXPORT_FILENAME_A = "trail_a"
EXPORT_FILENAME_B = "trail_b"
EXPORT_DPI = 300
EXPORT_HEIGHT = 720
EXPORT_WIDTH = 1280

assert captured_cam_a, "No captured camera for Checkpoint A -- click 'Capture Camera' first"
assert captured_cam_b, "No captured camera for Checkpoint B -- click 'Capture Camera' first"


def render_captured(trail_model, trail_data, captured_cam, height, width):
    renderer = mujoco.Renderer(trail_model, height=height, width=width)
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.azimuth = captured_cam["azimuth"]
    cam.elevation = captured_cam["elevation"]
    cam.distance = captured_cam["distance"]
    cam.lookat[:] = captured_cam["lookat"]
    renderer.update_scene(trail_data, camera=cam)
    return renderer.render()


def save_frame(frame, export_dir, filename, dpi):
    export_dir.mkdir(parents=True, exist_ok=True)
    for ext in ["svg", "pdf", "png"]:
        out_path = export_dir / f"{filename}.{ext}"
        fig, ax = plt.subplots(figsize=(frame.shape[1] / dpi, frame.shape[0] / dpi), dpi=dpi)
        ax.imshow(frame)
        ax.axis("off")
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        fig.savefig(out_path, dpi=dpi, bbox_inches="tight", pad_inches=0)
        plt.close(fig)
        print(f"Saved: {out_path}")


# --- Checkpoint A ---
frame_a = render_captured(trail_model_a, trail_data_a, captured_cam_a, EXPORT_HEIGHT, EXPORT_WIDTH)
save_frame(frame_a, EXPORT_DIR, EXPORT_FILENAME_A, EXPORT_DPI)

# --- Checkpoint B ---
frame_b = render_captured(trail_model_b, trail_data_b, captured_cam_b, EXPORT_HEIGHT, EXPORT_WIDTH)
save_frame(frame_b, EXPORT_DIR, EXPORT_FILENAME_B, EXPORT_DPI)


Saved: /home/mila/a/aidan.sirbu/SCAMPER/notebooks/plots/random_generation/trail_a.svg
Saved: /home/mila/a/aidan.sirbu/SCAMPER/notebooks/plots/random_generation/trail_a.pdf
Saved: /home/mila/a/aidan.sirbu/SCAMPER/notebooks/plots/random_generation/trail_a.png
Saved: /home/mila/a/aidan.sirbu/SCAMPER/notebooks/plots/random_generation/trail_b.svg
Saved: /home/mila/a/aidan.sirbu/SCAMPER/notebooks/plots/random_generation/trail_b.pdf
Saved: /home/mila/a/aidan.sirbu/SCAMPER/notebooks/plots/random_generation/trail_b.png
